In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import tiktoken
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from dataclasses import dataclass
import re


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cuda


In [2]:
@dataclass
class KVCache:
    K : torch.Tensor
    V : torch.Tensor

    def sequence_length(self) -> int:
        return self.K.shape[2]
    
    def update(self, new_k: torch.Tensor, new_v: torch.Tensor):
        self.K = torch.cat([self.K, new_k], dim=-2)
        self.V = torch.cat([self.V, new_v], dim=-2)

class RotaryPositionalEmbeddings(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, offset=0):
        B, N, H, D = x.shape
        t = torch.arange(offset, offset + N, device=x.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos()[None, :, None, :]   # 1, N, 1, D
        sin = emb.sin()[None, :, None, :]
        return x * cos + self._rotate_half(x) * sin

    def _rotate_half(self, x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

class MHA(nn.Module):
    def __init__(self, d_model, num_heads, is_causal = True, dropout = 0.2):
        assert d_model % num_heads == 0
        super().__init__()

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.num_heads = num_heads
        self.d_head = int(d_model / num_heads)
        self.scale = self.d_head ** 0.5
        self.causal = is_causal

        self.rope = RotaryPositionalEmbeddings(self.d_head)

    def forward(self, x, attn_mask = None, cache : KVCache | None = None):
        B, N, D = x.shape
        QKV = self.Wqkv(x)
        QKV = QKV.view(B, N, 3, self.num_heads, self.d_head)
        QKV = QKV.permute(2, 0, 3, 1, 4)
        Q, K, V = QKV[0], QKV[1], QKV[2] # B H N D

        offset = 0 if cache is None else cache.sequence_length()

        Q = Q.permute(0, 2, 1, 3)
        Q = self.rope(Q, offset).permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        K = self.rope(K, offset).permute(0, 2, 1, 3)

        if cache is None:
            cache = KVCache(K=K, V=V)
        else:
            cache.update(new_k=K, new_v=V)

        K = cache.K
        V = cache.V

        score = Q @ K.transpose(-2, -1) # batch H N S
        score = score / self.scale

        if attn_mask is not None:
            score = score.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))

        if self.causal:
            mask = torch.triu(torch.ones(Q.shape[-2], K.shape[-2], device=Q.device), diagonal= K.shape[-2] - Q.shape[-2] + 1).bool()
            score = score.masked_fill(mask, float('-inf'))

        weights = F.sigmoid(score)
        out = torch.matmul(weights, V) # batch H N d_head

        out = out.transpose(1, 2).contiguous() # B N H d_head
        out = out.view(B, N, -1)
        return self.Wo(out), cache

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, heads, dropout = 0.2):
        super().__init__()
        self.attn = MHA(d_model, heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, cache = None, attn_mask = None):
        attended, cache = self.attn(self.ln1(x), cache = cache, attn_mask = attn_mask)
        x = x + self.dropout(attended)
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x, cache

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformers = nn.ModuleList(
            [TransformerBlock(d_model, d_ff, num_heads) for i in range(num_layers)]
        )
        self.ln = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, vocab_size, bias = False)
        self.unembed.weight = self.embed.weight

    def forward(self, x, attn_mask = None):
        out = self.embed(x)
        for t in self.transformers:
            out, _ = t(out, attn_mask=attn_mask)
        norm = self.ln(out)
        pred = self.unembed(norm)

        return pred

    @torch.no_grad()
    def step(self, x, caches=None):
        out = self.embed(x)
        if caches is None:
            caches = [None] * len(self.transformers)
        new_caches = []
        for t, cache in zip(self.transformers, caches):
            out, cache = t(out, attn_mask=None, cache=cache)
            new_caches.append(cache)
        norm = self.ln(out)
        pred = self.unembed(norm)
        return pred, new_caches

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, live = True, caches = None, temperature = 2.0):
        x = input_ids
        generated = input_ids
        for i in range(max_new_tokens):
            logits, caches = self.step(x, caches)[:, -1, :] # B 1 D
            logits = logits / temperature
            probs = F.softmax(logits)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim = 1)
            x = next_token

        return generated

## Datasets:

First, we use the tinyshakespeare dataset at a character level for model validation. When learned, the model can generate Shakespeare-like writing, sometimes making up words, but coherent enough to replicate the structure and voice.

Next, we use the `gpt-2` tokenizer to read through wikipedia data. The adidtional tokens are added to help the decoder parse the dataset's special tokens. The context window is embedded into the format of the dataset.

In [3]:
seq_size = 128

text = open("data/tinyshakespeare.txt", "r").read()

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {c : i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)} 

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

def compile(x, seq_len):
        x_sequences = []
        y_sequences = []

        for i in tqdm(range(0, len(x) - seq_len - 1, int(seq_len/2))):
            x_sequences.append(x[i:i+seq_len])
            y_sequences.append(x[i+1:i+seq_len+1])

        x_sequences = np.array(x_sequences)
        y_sequences = np.array(y_sequences)

        return torch.tensor(x_sequences), torch.tensor(y_sequences)

tokens = encode(text)
x_data, y_data = compile(tokens, seq_size)
print(x_data.shape)
assert x_data.shape == y_data.shape, f"make sure x shape {x_data.shape} matches y shape {y_data.shape}"

print(x_data.shape[0])

split = int(0.8 * x_data.shape[0])
x_train, y_train = x_data[:split], y_data[:split]
x_val, y_val = x_data[split:], y_data[split:]

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

100%|██████████| 17427/17427 [00:00<00:00, 59928.80it/s]


torch.Size([17427, 128])
17427


In [ ]:
def compile(x, seq_len):
        x_sequences = []
        y_sequences = []

        for i in tqdm(range(0, len(x) - seq_len - 1, int(seq_len))):
            x_sequences.append(x[i:i+seq_len])
            y_sequences.append(x[i+1:i+seq_len+1])

        x_sequences = np.array(x_sequences)
        y_sequences = np.array(y_sequences)

        return torch.tensor(x_sequences), torch.tensor(y_sequences)

seq_size = 128
dl = False

base_enc = tiktoken.get_encoding("gpt2")

custom_tokens = {
    "<USER>",
    "<ASSISTANT>",
    "_START_ARTICLE_",
    "_START_SECTION_",
    "_START_PARAGRAPH_"
}

custom_token_ids = {
    token: base_enc.n_vocab + i for i, token in enumerate(custom_tokens)
}

enc = tiktoken.Encoding(
    name="gpt2_custom",
    pat_str=base_enc._pat_str,
    mergeable_ranks=base_enc._mergeable_ranks,
    special_tokens={**base_enc._special_tokens, **custom_token_ids},
)

special_tokens_set = set(custom_tokens) | {"<|endoftext|>"}

encode = enc.encode
decode = enc.decode
vocab_size = enc.n_vocab
print(vocab_size)

if dl:
    wiki = load_dataset("wiki40b", "en", split="train", streaming=True)
    with open("datasets/wiki.txt", "w") as f:
        for i, example in enumerate(wiki):
            if i >= 100000:
                break
            text = example['text']
            text = text.replace("_START_ARTICLE_", "")
            text = text.replace("_START_SECTION_", "")
            text = text.replace("_START_PARAGRAPH_", "")
            text = text.replace("_NEWLINE_", "\n")
            text = re.sub(r'\n+', '\n', text)
            text = re.sub(r' +', ' ', text)
            # strip leading title line
            lines = text.strip().split('\n')
            text = '\n'.join(lines[1:]).strip()   # ← skip first line
            if text:
                f.write(text + "\n")


text = open("datasets/wiki.txt", "r", encoding="utf-8").read()
text = re.sub(r'<[^>]+>', '', text)        # strip any html tags
text = re.sub(r'\s+', ' ', text)           # normalize whitespace
text = text.strip()

tokens = encode(text, allowed_special=special_tokens_set)
x_data, y_data = compile(tokens, seq_size)
print(decode(x_data[3].tolist()))
print(decode(y_data[3].tolist()))
assert x_data.shape == y_data.shape, f"make sure x shape {x_data.shape} matches y shape {y_data.shape}"

split = int(0.8 * x_data.shape[0])
x_train, y_train = x_data[:split], y_data[:split]
x_val, y_val = x_data[split:], y_data[split:]

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

## Model Training

In [4]:
epochs = 200
warmup_steps = 50

whereswaldo = Decoder(
    vocab_size=vocab_size,
    d_model=32,
    num_heads=8,
    d_ff=128,
    num_layers=12,
).to(device)

print(f"Parameters: {sum(p.numel() for p in whereswaldo.parameters())}")
print(f"embed params:      {sum(p.numel() for n, p in whereswaldo.named_parameters() if 'embed' in n):,}")
print(f"transformer params:{sum(p.numel() for n, p in whereswaldo.named_parameters() if 'transformer' in n):,}")

print(f"{len(tokens):,} tokens")
print(f"ratio: {len(tokens) / sum(p.numel() for p in whereswaldo.parameters()):.2f}x")

optimizer = torch.optim.AdamW(whereswaldo.parameters(), lr=3e-4, weight_decay=0.1)
warmup = LinearLR(
    optimizer, 
    start_factor=0.01,    # starts at 1% of lr
    end_factor=1.0,       # ramps up to full lr
    total_iters=warmup_steps       # over 200 steps
)

cosine = CosineAnnealingLR(
    optimizer,
    T_max=epochs * len(train_loader) - warmup_steps,  # remaining steps after warmup
    eta_min=3e-5          # minimum lr at end
)

nn.init.normal_(whereswaldo.embed.weight, std=0.02)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])
loss_fn = F.cross_entropy

Parameters: 154592
embed params:      2,080
transformer params:152,448
1,115,394 tokens
ratio: 7.22x


In [5]:
train_losses = []
val_losses = []
for i in range(epochs):
    whereswaldo.train()
    train_loss = 0
    for train_batch in tqdm(train_loader):
        # with autocast(str(device)):
        x, y = train_batch
        x = x.to(device)
        y = y.to(device)
        pred = whereswaldo(x)
        loss = loss_fn(pred.view(-1, vocab_size), y.view(-1))

        # scaler.scale(loss).backward()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(whereswaldo.parameters(), max_norm=1.0)
        # scaler.step(optimizer)
        optimizer.step()
        # scaler.update()
        scheduler.step()

        train_loss += loss
    train_loss /= len(train_loader)
    
    whereswaldo.eval()
    with torch.no_grad():
        val_loss = 0
        for val_batch in tqdm(val_loader):
            x, y = val_batch
            x = x.to(device)
            y = y.to(device)
            pred = whereswaldo(x)
            val_loss += loss_fn(pred.view(-1, vocab_size), y.view(-1))
    
    val_loss /= len(val_loader)

    print(f"Epoch {i+1}: train loss = {train_loss:.6f}, val loss = {val_loss:.6f}")
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot()

ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Validation")
ax.legend()
ax.set_title("Training vs Validation Loss")
plt.show()


100%|██████████| 28/28 [00:01<00:00, 16.18it/s]


Epoch 1: train loss = 3.855263, val loss = 3.526374


100%|██████████| 28/28 [00:01<00:00, 16.01it/s]


Epoch 2: train loss = 3.405628, val loss = 3.365235


100%|██████████| 28/28 [00:02<00:00, 11.65it/s]


...

Epoch 16: train loss = 2.445847, val loss = 2.420743


100%|██████████| 28/28 [00:02<00:00, 12.35it/s]


Epoch 17: train loss = 2.432406, val loss = 2.409287


 21%|██▏       | 6/28 [00:07<00:27,  1.26s/it]


KeyboardInterrupt: 

In [ ]:
save = True
load = False

if save:
    torch.save({
        'model_state': whereswaldo.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epoch': i,
        'val_loss': val_loss,
    }, 'whereswaldo.pt')


if load:
    checkpoint = torch.load('whereswaldo.pt')
    whereswaldo.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])

In [ ]:
def query(model : Decoder, prompt, generation_len = 2 * seq_size, live = True):
    model.eval()
    tokens = torch.tensor(encode(prompt)).unsqueeze(0).to(device)
    if live: print(prompt, end="")


    for i in range(generation_len):
        tokens_cropped = tokens[:, -seq_size:]
        scores = model(tokens_cropped)[:, -1, :]
        probs = F.softmax(scores, dim = -1)
        char = torch.multinomial(probs, num_samples=1)
        if live: print(decode([char[0].item()]), end = "") 
        tokens = torch.cat([tokens, char], dim = 1)

    return decode(tokens[0].tolist())

res = query(whereswaldo, "ROMEO:", 2000)